# Gate 4b: QLoRA students (train, then dev rewrites)

Rules: `docs/ROADMAP.md`, gate 4b. One configuration for all three models (`scripts/qlora.py`, `CONFIG`):
4-bit NF4 base, LoRA r 16 on all linear layers, 3 epochs on the 966 training pairs, loss on the rewrite only.
The adapter after the last epoch is merged into the bf16 base and rewrites the 50 dev questions with exactly the
gate 4a decoding. Retrieval is scored locally afterwards with `scripts/eval_rewrites.py`.

Everything is written to Google Drive (`MyDrive/kvkk-rewriter/`) as soon as it exists. If the runtime
disconnects, run all again: finished models are skipped, and a model whose adapter is saved is not retrained.

Before running:
1. Runtime -> Change runtime type -> **L4 GPU**.
2. Secrets (key icon) -> `HF_TOKEN` (Hugging Face read token with Llama-3.1 access), notebook access on.
3. Runtime -> Run all, and allow Google Drive access when asked. Keep the tab open.
4. At the end a zip with the result files downloads (adapters stay on Drive). Add the row printed by the last
   cell to `results/colab_usage.csv`, then disconnect and delete the runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert "L4" in gpu, f"expected an L4 runtime (docs/ROADMAP.md, gate 4b), got {gpu!r}"

In [ ]:
import os
if not os.path.exists("/content/kvkk-rewriter"):
    subprocess.run(["git", "clone", "-q", "--recurse-submodules",
                    "https://github.com/EmirhanHasirci11/kvkk-rewriter.git", "/content/kvkk-rewriter"], check=True)
%cd /content/kvkk-rewriter
!git pull -q && git log -1 --format='%H %s'
!pip install -q "transformers>=4.56" accelerate peft bitsandbytes
# Colab's preinstalled torchao is older than peft accepts, and loading the adapter then fails; nothing here uses it.
!pip uninstall -q -y torchao

In [ ]:
import json, sys, time
from pathlib import Path

import torch, transformers, peft, bitsandbytes
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/kvkk-rewriter")
(DRIVE / "results").mkdir(parents=True, exist_ok=True)
(DRIVE / "adapters").mkdir(parents=True, exist_ok=True)

sys.path.insert(0, "scripts")
import qlora, student

TAG = "qlora_v1"
ORDER = ["kumru-2b", "qwen3-4b", "llama3.1-8b"]    # fastest first
HF_TOKEN = userdata.get("HF_TOKEN")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
dev = [json.loads(l) for l in Path("external/kvkk-rag/eval/golden.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
warmup = qlora.load_pairs("train")[0]["question"]
LOG = DRIVE / f"results/colab_{TAG}.log"
versions = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__}


def log(msg):
    line = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


log(f"start: {gpu}, commit {COMMIT[:7]}, {versions}, config {qlora.CONFIG}")
print(len(dev), "dev questions")

In [ ]:
t_start = time.time()
for name in ORDER:
    hf_id = student.MODELS[name]
    adapter = DRIVE / f"adapters/{TAG}_{name}"
    out = DRIVE / f"results/rewrites_{TAG}_{name}_dev_v1.jsonl"
    if out.exists():
        log(f"=== {name}: {out.name} already on Drive, skipped")
        continue
    if (adapter / "train_record.json").exists():
        log(f"=== {name}: adapter already on Drive, not retrained")
    else:
        log(f"=== {name}: training ({hf_id})")
        record = qlora.train(hf_id, adapter, token=HF_TOKEN, log=log)
        record.update({"gpu": gpu, "commit": COMMIT, **versions})
        (adapter / "train_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
        log(f"{name}: trained in {record['train_min']} min, val loss {record['val_loss']}, peak {record['peak_gpu_gb']} GB")
    torch.cuda.empty_cache()

    model, tok, stop = qlora.load_merged(hf_id, adapter, token=HF_TOKEN)
    torch.cuda.reset_peak_memory_stats()
    student.rewrite(model, tok, stop, warmup)
    rows = []
    for i, q in enumerate(dev, 1):
        r = student.rewrite(model, tok, stop, q["question"])
        rows.append({"id": q["id"], "question": q["question"], **r, "model": name, "hf_id": hf_id, "adapter": adapter.name})
        print(f"[{i:2}/{len(dev)}] {r['latency_s']:.2f}s {r['new_tokens']:3} tok{' TRUNC' if r['truncated'] else ''}  {r['rewrite'][:110]}")
    out.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    lat = sorted(r["latency_s"] for r in rows)
    log(f"{name}: dev rewrites written; latency p50 {lat[len(lat) // 2]} s, truncated {sum(r['truncated'] for r in rows)}, "
        f"inference peak {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    del model
    torch.cuda.empty_cache()
log(f"done in {(time.time() - t_start) / 60:.1f} min this session")

In [ ]:
import zipfile
from google.colab import files
paths = sorted((DRIVE / "results").glob(f"rewrites_{TAG}_*_dev_v1.jsonl")) + [LOG]
records = sorted((DRIVE / "adapters").glob(f"{TAG}_*/train_record.json"))
with zipfile.ZipFile(f"{TAG}.zip", "w") as z:
    for p in paths:
        z.write(p, f"results/{p.name}")
    for p in records:
        z.write(p, f"results/train_record_{p.parent.name}.json")
files.download(f"{TAG}.zip")
print("row for results/colab_usage.csv (fill in runtime_min and compute_units from Colab):")
print(f"{time.strftime('%Y-%m-%d')},notebooks/02_qlora.ipynb,{gpu},<runtime_min>,<compute_units>,QLoRA training + dev rewrites ({len(records)} models),commit {COMMIT[:7]}")